In [1]:
import pandas as pd
import numpy as np

from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate, GridSearchCV
from surprise import accuracy
from collections import defaultdict

In [2]:
# TCVファイルの読み込み
df_train = pd.read_csv('../000.data/train/train_B.tsv', sep="\t")
test = pd.read_csv('../000.data/test.tsv', sep="\t")

In [3]:
# 末尾が "_B" のテストデータだけを抽出
df_test = test[test["user_id"].str.endswith("_B")]

In [4]:
# 広告経由の購入のみ関連度3、それ以外はスコアをそのまま使用
def calculate_relevance(row):
    if row['event_type'] == 3 and row['ad'] == 1:
        return 3
    return row['event_type']

df_train['rating'] = df_train.apply(calculate_relevance, axis=1)

In [5]:
# Surprise用データセット形式に変換
reader = Reader(rating_scale=(0, 3))
data = Dataset.load_from_df(df_train[['user_id', 'product_id', 'rating']], reader)

In [6]:
# ハイパーパラメータの自動調整
param_grid = {
    'n_factors': [50, 100],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.02, 0.1]
}
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3)
gs.fit(data)

# 最適パラメータでモデル作成
print(f"Best RMSE Score: {gs.best_score['rmse']}")
print(f"Best Parameters: {gs.best_params['rmse']}")
model = SVD(**gs.best_params['rmse'])

Best RMSE Score: 0.36190758676254825
Best Parameters: {'n_factors': 50, 'lr_all': 0.002, 'reg_all': 0.1}


In [7]:
# nDCG計算
def calculate_ndcg(predictions, k=22):
    def dcg(scores):
        return sum([score / np.log2(idx + 2) for idx, score in enumerate(scores)])

    user_ranking = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_ranking[uid].append((iid, est))

    ndcg_values = []
    for uid, user_preds in user_ranking.items():
        user_preds.sort(key=lambda x: x[1], reverse=True)
        relevance_scores = [3 if i < k else 0 for i, _ in enumerate(user_preds)]
        ideal_scores = sorted(relevance_scores, reverse=True)
        ndcg = dcg(relevance_scores) / dcg(ideal_scores) if dcg(ideal_scores) > 0 else 0
        ndcg_values.append(ndcg)

    return np.mean(ndcg_values)

In [8]:
# nDCGの評価を実施
full_trainset = data.build_full_trainset()
model.fit(full_trainset)
testset = full_trainset.build_testset()
predictions = model.test(testset)
print(f"nDCG Score: {calculate_ndcg(predictions):.4f}")

nDCG Score: 1.0000


In [9]:
# 提出用データ作成
def create_submission(model, user_ids, k=22):
    result = []
    for uid in user_ids:
        all_predictions = [model.predict(uid, iid) for iid in df_train['product_id'].unique()]
        all_predictions.sort(key=lambda x: x.est, reverse=True)
        top_k = all_predictions[:k]
        result.extend([[uid, pred.iid, rank + 1] for rank, pred in enumerate(top_k)])
    return pd.DataFrame(result, columns=['user_id', 'product_id', 'rank'])

In [10]:
user_ids = df_test['user_id'].unique()
submission_df = create_submission(model, user_ids)

In [11]:
submission_df.to_csv('./predict_file/recommendation_result_B.tsv', sep="\t", index=False)